# Phase 5 — Évaluer ton modèle, puis le publier

Un modèle qui « a l'air » d'écrire bien, ça ne suffit pas : il faut des **chiffres**. On va mesurer trois choses :
1. **la perplexité** sur tout le jeu de validation, comparée à des modèles simples (dont l'héritier de ton bigramme de la phase 0) ;
2. **la grammaire** : préfère-t-il « les enfants mangent » à « les enfants mange » ?
3. **les connaissances** : sait-il que la capitale de l'Italie est Rome ?

Puis on le **publie sur Hugging Face**, avec une fiche qui donne ces résultats.

**Type d'exécution : GPU T4 si tu en as, sinon CPU.** Sur CPU, tout marche, mais la perplexité est calculée sur un échantillon de 150 000 tokens au lieu des 2,8 millions (sinon ça prendrait des heures). Compte une quinzaine de minutes en tout.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
DOSSIER = "/content/drive/MyDrive/petit-llm"

!pip -q install tokenizers safetensors huggingface_hub
import os, sys, math, json, shutil
import numpy as np, torch, torch.nn.functional as F
from tokenizers import Tokenizer
sys.path.append(DOSSIER)
from model import Config, GPT

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16
config = Config()
tokenizer = Tokenizer.from_file(f"{DOSSIER}/tokenizer.json")

modele = GPT(config).to(device)
modele.load_state_dict(torch.load(f"{DOSSIER}/modele_final.pt", map_location=device))
modele.eval()

for nom in ["train.bin", "val.bin"]:
    if not os.path.exists(f"/content/{nom}"):
        shutil.copy(f"{DOSSIER}/{nom}", f"/content/{nom}")
train = np.memmap("/content/train.bin", dtype=np.uint16, mode="r")
val = np.memmap("/content/val.bin", dtype=np.uint16, mode="r")
print(f"Modèle chargé sur {device} | val : {len(val):,} tokens")

Mounted at /content/drive
Modèle chargé sur cpu | val : 2,770,472 tokens


## 1. La perplexité, sur TOUT le jeu de validation

Pendant l'entraînement, on n'évaluait que 40 lots tirés au hasard. Ici, sur GPU, on passe sur les **2,8 millions de tokens** de `val.bin`, découpés en morceaux de 512. Sur CPU, on se limite aux 300 premiers morceaux (~150 000 tokens) : c'est une estimation, mais déjà précise à quelques centièmes près.

**La perplexité** = e^perte. C'est le nombre moyen de tokens entre lesquels le modèle hésite. Plus c'est bas, mieux c'est.

In [2]:
T = config.block_size
N_MORCEAUX = None if device == "cuda" else 300       # None = tout le jeu de validation
n_total = (len(val) - 1) // T
n_eval = n_total if N_MORCEAUX is None else min(N_MORCEAUX, n_total)
val_eval = val[: n_eval * T + 1]                     # les mêmes tokens serviront aux comparaisons

@torch.no_grad()
def perte_gpt(donnees, lot=32 if device == "cuda" else 8):
    n = (len(donnees) - 1) // T
    total, compte = 0.0, 0
    for debut in range(0, n, lot):
        idx = range(debut, min(debut + lot, n))
        x = torch.stack([torch.from_numpy(donnees[i*T:i*T+T].astype(np.int64)) for i in idx]).to(device)
        y = torch.stack([torch.from_numpy(donnees[i*T+1:i*T+T+1].astype(np.int64)) for i in idx]).to(device)
        with torch.autocast(device, dtype=dtype, enabled=(device == "cuda")):
            _, perte = modele(x, y)
        total += perte.item() * x.numel()
        compte += x.numel()
    return total / compte

perte_val = perte_gpt(val_eval)
print(f"Évalué sur {n_eval * T:,} tokens")
print(f"Perte de validation : {perte_val:.3f} | perplexité : {math.exp(perte_val):.1f}")

Évalué sur 153,600 tokens
Perte de validation : 2.699 | perplexité : 14.9


## 2. Contre qui se compare-t-on ?

Un chiffre seul ne veut rien dire. On calcule la perte de trois modèles de référence, **sur les mêmes tokens** :
- **le hasard** : chaque token a la même probabilité ;
- **l'unigramme** : il connaît la fréquence de chaque token, mais ne regarde **aucun** contexte ;
- **le bigramme** : ton modèle de la phase 0, mais sur les tokens au lieu des caractères. Il regarde **1 token** en arrière.

Ton GPT regarde jusqu'à **512 tokens** en arrière, et ses vecteurs partagent ce qu'ils apprennent entre contextes semblables. L'écart entre le bigramme et lui, c'est exactement ce que t'apportent l'attention et les embeddings.

On compte les bigrammes sur 100 M de tokens d'entraînement. Le lissage : quand une paire n'a jamais été vue, on se rabat en partie sur l'unigramme (sinon sa probabilité serait 0 et la perte infinie).

In [3]:
V = config.vocab_size
echantillon = np.asarray(train[:100_000_000]).astype(np.int64)

# Unigramme
comptes_uni = np.bincount(echantillon, minlength=V) + 1          # +1 : aucun token à probabilité nulle
p_uni = comptes_uni / comptes_uni.sum()

# Bigramme : chaque paire (a, b) devient un seul nombre a * V + b, qu'on compte
paires = echantillon[:-1] * V + echantillon[1:]
codes, comptes_paires = np.unique(paires, return_counts=True)
del paires

v = np.asarray(val_eval).astype(np.int64)
precedent, suivant = v[:-1], v[1:]

perte_hasard = math.log(V)
perte_uni = -np.log(p_uni[suivant]).mean()

pos = np.searchsorted(codes, precedent * V + suivant)
pos = np.minimum(pos, len(codes) - 1)
vu = codes[pos] == precedent * V + suivant
c_paire = np.where(vu, comptes_paires[pos], 0)
c_prec = comptes_uni[precedent] - 1
LAMBDA = 0.9
p_bi = np.where(c_prec > 0, c_paire / np.maximum(c_prec, 1), 0.0)
perte_bi = -np.log(LAMBDA * p_bi + (1 - LAMBDA) * p_uni[suivant]).mean()

print(f"{'Modèle':<28}{'Contexte':>12}{'Perte':>9}{'Perplexité':>13}")
print("-" * 62)
for nom, ctx, p in [("Hasard", "aucun", perte_hasard), ("Unigramme", "aucun", perte_uni),
                    ("Bigramme (phase 0)", "1 token", perte_bi), ("Ton GPT", "512 tokens", perte_val)]:
    print(f"{nom:<28}{ctx:>12}{p:>9.3f}{math.exp(p):>13,.1f}")

Modèle                          Contexte    Perte   Perplexité
--------------------------------------------------------------
Hasard                             aucun    9.704     16,384.0
Unigramme                          aucun    7.391      1,620.7
Bigramme (phase 0)               1 token    4.941        139.9
Ton GPT                       512 tokens    2.699         14.9


## 3. La grammaire : des paires minimales

C'est la méthode des benchmarks comme BLiMP, un test très linguistique. On donne au modèle deux phrases presque identiques, l'une correcte, l'autre fautive. On calcule la **probabilité de chaque phrase entière** : s'il donne plus de probabilité à la bonne, c'est un point.

Le modèle ne « juge » rien : on regarde seulement s'il trouve la phrase correcte plus probable.

Les paires avec **attracteur** sont les plus dures : dans « la population de ces villes augmente », le pluriel « villes » est collé au verbe, mais le sujet est « la population ». Un modèle qui ne regarde que le mot d'avant tombe dans le piège.

In [4]:
@torch.no_grad()
def logprob_phrase(texte):
    """Somme des log-probabilités de chaque token de la phrase (sauf le premier)."""
    ids = torch.tensor([tokenizer.encode(texte).ids], device=device)
    logits, _ = modele(ids)
    lp = F.log_softmax(logits[0, :-1].float(), dim=-1)
    return lp.gather(1, ids[0, 1:, None]).sum().item()

paires_minimales = [
    ("accord sujet-verbe", "Les enfants mangent une pomme.", "Les enfants mange une pomme."),
    ("accord sujet-verbe", "Je ne sais pas pourquoi.", "Je ne sait pas pourquoi."),
    ("accord sujet-verbe", "Ils ont construit un pont.", "Ils a construit un pont."),
    ("accord sujet-verbe", "Nous avons fini le travail.", "Nous avez fini le travail."),
    ("accord sujet-verbe", "Elle se lave les mains.", "Elle se lavent les mains."),
    ("accord adjectif", "Les maisons sont grandes.", "Les maisons sont grands."),
    ("accord adjectif", "Ces églises sont anciennes.", "Ces églises sont anciens."),
    ("accord déterminant", "Paris est une grande ville.", "Paris est un grande ville."),
    ("auxiliaire", "Je me suis levé tôt.", "Je m'ai levé tôt."),
    ("participe / infinitif", "Il a mangé une pomme.", "Il a manger une pomme."),
    ("subjonctif", "Il faut que tu viennes demain.", "Il faut que tu viens demain."),
    ("sujet coordonné", "Le roi et la reine sont arrivés.", "Le roi et la reine est arrivés."),
    ("attracteur", "Les élèves de la classe travaillent.", "Les élèves de la classe travaille."),
    ("attracteur", "La population de ces villes augmente.", "La population de ces villes augmentent."),
    ("attracteur", "Le directeur des écoles est parti.", "Le directeur des écoles sont partis."),
    ("relative", "C'est la ville où il est né.", "C'est la ville que il est né."),
]

bons = 0
for categorie, correcte, fautive in paires_minimales:
    ok = logprob_phrase(correcte) > logprob_phrase(fautive)
    bons += ok
    print(f"{'✅' if ok else '❌'} [{categorie}] {correcte}  /  {fautive}")
score_grammaire = bons / len(paires_minimales)
print(f"\nGrammaire : {bons}/{len(paires_minimales)} ({score_grammaire:.0%}) | hasard : 50 %")

✅ [accord sujet-verbe] Les enfants mangent une pomme.  /  Les enfants mange une pomme.
❌ [accord sujet-verbe] Je ne sais pas pourquoi.  /  Je ne sait pas pourquoi.
✅ [accord sujet-verbe] Ils ont construit un pont.  /  Ils a construit un pont.
✅ [accord sujet-verbe] Nous avons fini le travail.  /  Nous avez fini le travail.
✅ [accord sujet-verbe] Elle se lave les mains.  /  Elle se lavent les mains.
✅ [accord adjectif] Les maisons sont grandes.  /  Les maisons sont grands.
✅ [accord adjectif] Ces églises sont anciennes.  /  Ces églises sont anciens.
✅ [accord déterminant] Paris est une grande ville.  /  Paris est un grande ville.
✅ [auxiliaire] Je me suis levé tôt.  /  Je m'ai levé tôt.
✅ [participe / infinitif] Il a mangé une pomme.  /  Il a manger une pomme.
❌ [subjonctif] Il faut que tu viennes demain.  /  Il faut que tu viens demain.
✅ [sujet coordonné] Le roi et la reine sont arrivés.  /  Le roi et la reine est arrivés.
❌ [attracteur] Les élèves de la classe travaillent.  /  Les él

## 4. Les connaissances : un QCM

Même principe : on donne un début de phrase et plusieurs fins possibles, et on regarde laquelle le modèle trouve la plus probable. Hasard : 25 % (4 choix).

**Piège de mesure** : « Ouagadougou » fait plusieurs tokens, « Dakar » moins. Si on **additionne** les log-probabilités, chaque token en plus coûte des points, et les réponses longues sont pénalisées. Si on prend la **moyenne par token**, ce biais disparaît, mais un autre apparaît. On affiche les deux. Les vrais benchmarks font pareil (`acc` et `acc_norm`).

In [5]:
@torch.no_grad()
def logprob_suite(debut, suite):
    """Log-probabilité de `suite` sachant `debut` : somme et moyenne par token."""
    ids_debut = tokenizer.encode(debut).ids
    ids_suite = tokenizer.encode(debut + suite).ids[len(ids_debut):]
    ids = torch.tensor([ids_debut + ids_suite], device=device)
    logits, _ = modele(ids)
    lp = F.log_softmax(logits[0, :-1].float(), dim=-1)
    cibles = ids[0, 1:]
    lp_suite = lp.gather(1, cibles[:, None])[-len(ids_suite):]
    return lp_suite.sum().item(), lp_suite.mean().item()

qcm = [
    ("La capitale de la France est", " Paris", [" Berlin", " Madrid", " Rome"]),
    ("La capitale de l'Italie est", " Rome", [" Paris", " Madrid", " Lisbonne"]),
    ("La capitale de l'Espagne est", " Madrid", [" Rome", " Lisbonne", " Berlin"]),
    ("La capitale du Japon est", " Tokyo", [" Pékin", " Séoul", " Bangkok"]),
    ("La capitale du Sénégal est", " Dakar", [" Bamako", " Abidjan", " Niamey"]),
    ("La capitale du Burkina Faso est", " Ouagadougou", [" Dakar", " Bamako", " Niamey"]),
    ("La tour Eiffel se trouve à", " Paris", [" Londres", " Rome", " Berlin"]),
    ("Victor Hugo est un écrivain", " français", [" allemand", " italien", " russe"]),
    ("Mozart est un compositeur", " autrichien", [" français", " espagnol", " russe"]),
    ("Le Nil est un fleuve d'", "Afrique", ["Asie", "Europe", "Amérique"]),
    ("La langue officielle du Brésil est le", " portugais", [" espagnol", " français", " anglais"]),
    ("Le plus grand océan du monde est l'océan", " Pacifique", [" Atlantique", " Indien", " Arctique"]),
    ("Le Rhône se jette dans la mer", " Méditerranée", [" du Nord", " Baltique", " Noire"]),
    ("La Révolution française commence en", " 1789", [" 1689", " 1889", " 1589"]),
    ("Napoléon Bonaparte est né en", " Corse", [" Bretagne", " Normandie", " Alsace"]),
    ("La Joconde est exposée au musée du", " Louvre", [" Prado", " Vatican", " Caire"]),
]

score_somme, score_moyenne = 0, 0
for debut, bonne, mauvaises in qcm:
    choix = [bonne] + mauvaises
    lps = [logprob_suite(debut, c) for c in choix]
    pred_somme = choix[int(np.argmax([s for s, _ in lps]))]
    pred_moy = choix[int(np.argmax([m for _, m in lps]))]
    score_somme += pred_somme == bonne
    score_moyenne += pred_moy == bonne
    print(f"{debut}…  attendu :{bonne} | somme :{pred_somme} {'✅' if pred_somme == bonne else '❌'}"
          f" | moyenne :{pred_moy} {'✅' if pred_moy == bonne else '❌'}")

n = len(qcm)
print(f"\nConnaissances : somme {score_somme}/{n} ({score_somme/n:.0%}) | "
      f"moyenne {score_moyenne}/{n} ({score_moyenne/n:.0%}) | hasard : 25 %")

La capitale de la France est…  attendu : Paris | somme : Paris ✅ | moyenne : Paris ✅
La capitale de l'Italie est…  attendu : Rome | somme : Rome ✅ | moyenne : Rome ✅
La capitale de l'Espagne est…  attendu : Madrid | somme : Madrid ✅ | moyenne : Madrid ✅
La capitale du Japon est…  attendu : Tokyo | somme : Tokyo ✅ | moyenne : Bangkok ❌
La capitale du Sénégal est…  attendu : Dakar | somme : Dakar ✅ | moyenne : Niamey ❌
La capitale du Burkina Faso est…  attendu : Ouagadougou | somme : Ouagadougou ✅ | moyenne : Ouagadougou ✅
La tour Eiffel se trouve à…  attendu : Paris | somme : Paris ✅ | moyenne : Paris ✅
Victor Hugo est un écrivain…  attendu : français | somme : français ✅ | moyenne : français ✅
Mozart est un compositeur…  attendu : autrichien | somme : français ❌ | moyenne : français ❌
Le Nil est un fleuve d'…  attendu :Afrique | somme :Afrique ✅ | moyenne :Afrique ✅
La langue officielle du Brésil est le…  attendu : portugais | somme : portugais ✅ | moyenne : portugais ✅
Le plus grand o

## 5. La température

À chaque token, le modèle donne une probabilité à tous les tokens possibles, et on en **tire** un. La température change ce tirage :
- **basse (0,3)** : on choisit presque toujours le plus probable → texte sûr mais répétitif ;
- **haute (1,5)** : on aplatit les probabilités → texte varié mais incohérent.

Même modèle, mêmes poids : seule la manière de tirer change.

In [6]:
debut = torch.tensor([tokenizer.encode("Le fleuve").ids], device=device)
for t in [0.3, 0.8, 1.5]:
    torch.manual_seed(1)
    sortie = modele.generer(debut, 40, temperature=t)
    print(f"--- température {t} ---\n{tokenizer.decode(sortie[0].tolist())}\n")

--- température 0.3 ---
Le fleuve a été créé en 1974.

Géographie 
Le fleuve prend sa source dans la chaîne de la rivière des Outaouais, dans le territoire non organisé de Lac-Ashuap

--- température 0.8 ---
Le fleuve a deux cours d'eau naturels :
 le ruisseau de la Rambouanne (rd) dans le canton de Saint-Omer.
 le ruisseau de Cœur (rd) dans le

--- température 1.5 ---
Le fleuve a deux cours d’eau naturels : bassin amèneissance écologique du bassin versant du Rhône (en tenant compte, des’Ouleilles, du Doubs ; bassin hydraulique),



## 6. Publier sur Hugging Face

1. Va sur **huggingface.co/settings/tokens** → *Create new token* → type **Write**. Copie-le.
2. Lance la cellule suivante et colle le token quand on te le demande.

On publie :
- `model.safetensors` : les poids (format sûr et standard) ;
- `config.json`, `model.py`, `tokenizer.json` : de quoi reconstruire le modèle ;
- `journal.csv` : la courbe d'entraînement ;
- `README.md` : la **fiche du modèle**, remplie automatiquement avec tes résultats.

Le dépôt est créé en **privé** : vérifie la fiche sur le site, puis passe `PRIVE = False` et relance la dernière cellule pour le rendre public.

In [7]:
from huggingface_hub import login
login()

In [8]:
from huggingface_hub import HfApi
from safetensors.torch import save_model
from dataclasses import asdict

NOM_DEPOT = "Ezeck01/petit-gpt-fr-46m"   # change le nom si tu veux
PRIVE = True
LICENCE = "cc-by-sa-3.0"                 # la licence de Wikipédia, les données d'entraînement

nb_params = sum(p.numel() for p in modele.parameters())
journal = f"{DOSSIER}/journal.csv"

fiche = f"""---
language: fr
license: {LICENCE}
library_name: pytorch
tags: [text-generation, gpt, from-scratch, french]
datasets: [wikimedia/wikipedia]
---

# Petit GPT français — {nb_params / 1e6:.0f} M de paramètres, entraîné de zéro

Modèle de langue de type GPT (décodeur Transformer), **entièrement construit et pré-entraîné de zéro** :
tokenizer, architecture, boucle d'entraînement et évaluation. Projet d'apprentissage.

## Architecture

| | |
|---|---|
| Paramètres | {nb_params:,} |
| Blocs | {config.n_layer} |
| Têtes d'attention | {config.n_head} |
| Dimension | {config.n_embd} |
| Contexte | {config.block_size} tokens |
| Vocabulaire | {config.vocab_size:,} (BPE byte-level, entraîné sur le corpus) |

Embeddings de position appris, LayerNorm avant chaque sous-couche, GELU, poids partagés entre l'embedding et la couche de sortie.

## Données

Wikipédia en français (`wikimedia/wikipedia`, dump `20231101.fr`), articles mélangés.
Sections finales retirées (Notes et références, Liens externes, Voir aussi, Bibliographie), articles de moins de 500 caractères écartés.
**{len(train):,} tokens** d'entraînement, **{len(val):,} tokens** de validation (1 article sur 200, jamais vus pendant l'entraînement).

## Entraînement

Un passage sur le corpus : 16 000 étapes de 32 768 tokens. AdamW (β = 0,9 / 0,95, weight decay 0,1), taux d'apprentissage 6e-4 avec warmup de 500 étapes puis décroissance en cosinus jusqu'à 6e-5, précision mixte float16. Environ 4 h sur un GPU T4 (Google Colab).

## Résultats

Perte mesurée sur {n_eval * T:,} tokens de validation.

| Modèle | Contexte | Perte val | Perplexité |
|---|---|---|---|
| Hasard | aucun | {perte_hasard:.3f} | {math.exp(perte_hasard):,.0f} |
| Unigramme | aucun | {perte_uni:.3f} | {math.exp(perte_uni):,.0f} |
| Bigramme lissé | 1 token | {perte_bi:.3f} | {math.exp(perte_bi):,.0f} |
| **Ce modèle** | 512 tokens | **{perte_val:.3f}** | **{math.exp(perte_val):.1f}** |

- **Grammaire** (paires minimales en français, style BLiMP) : {score_grammaire:.0%} ({len(paires_minimales)} paires, hasard 50 %)
- **Connaissances** (QCM, 4 choix) : {score_somme/n:.0%} en somme des log-probabilités, {score_moyenne/n:.0%} en moyenne par token ({n} questions, hasard 25 %)

Ces tests sont petits et faits maison : ils donnent un ordre de grandeur, pas un classement.

## Limites

- Modèle de **complétion**, pas un assistant : il continue un texte, il ne répond pas à des questions.
- Style encyclopédique : il écrit comme Wikipédia.
- Il **invente** des faits avec assurance (hallucinations).
- Les nombres et dates écrits avec des modèles wiki sont absents du corpus nettoyé : le modèle laisse souvent des trous (« peuplée de plus de  personnes »).

## Utilisation

```python
import torch, json
from huggingface_hub import hf_hub_download
from safetensors.torch import load_model
from tokenizers import Tokenizer

depot = "{NOM_DEPOT}"
for f in ["model.py", "config.json", "tokenizer.json", "model.safetensors"]:
    hf_hub_download(depot, f, local_dir=".")

from model import Config, GPT
modele = GPT(Config(**json.load(open("config.json"))))
load_model(modele, "model.safetensors")
modele.eval()

tok = Tokenizer.from_file("tokenizer.json")
ids = torch.tensor([tok.encode("La ville de").ids])
print(tok.decode(modele.generer(ids, 50, temperature=0.8)[0].tolist()))
```
"""

PUBLICATION = "/content/publication"
os.makedirs(PUBLICATION, exist_ok=True)
save_model(modele, f"{PUBLICATION}/model.safetensors")
json.dump(asdict(config), open(f"{PUBLICATION}/config.json", "w"), indent=2)
for f in ["model.py", "tokenizer.json", "journal.csv"]:
    shutil.copy(f"{DOSSIER}/{f}", f"{PUBLICATION}/{f}")
open(f"{PUBLICATION}/README.md", "w", encoding="utf-8").write(fiche)
print("Fichiers prêts :", os.listdir(PUBLICATION))

api = HfApi()
api.create_repo(NOM_DEPOT, private=PRIVE, exist_ok=True)
api.update_repo_settings(NOM_DEPOT, private=PRIVE)
api.upload_folder(folder_path=PUBLICATION, repo_id=NOM_DEPOT,
                  commit_message="Petit GPT français entraîné de zéro")
print(f"Publié ({'privé' if PRIVE else 'public'}) : https://huggingface.co/{NOM_DEPOT}")

Fichiers prêts : ['README.md', 'tokenizer.json', 'model.safetensors', 'config.json', 'model.py', 'journal.csv']
Publié (privé) : https://huggingface.co/Ezeck01/petit-gpt-fr-46m
